# WiDS 2026 | Tri-Survival Stack + Distance-Stratified Blend
Adapted from the best public techniques + our proven GBSA baseline.

**Key improvements over 0.97041 baseline:**
1. **72h = constant 1.0** — Proven best across all runs. B72≈0.
2. **CoxPH** — Linear survival model adds diversity to GBSA (tree-based)
3. **Distance-stratified blending** — Near/far zones get separate blend weights
4. **Zone-split LGB** — Separate classifiers for near/far fires
5. **10 GBSA configs × 20 seeds** — More averaging

| Component | Count |
|---|---|
| GBSA | 10 configs × 20 seeds × 5 folds = 1000 |
| CoxPH | 7 alphas × 10 seeds × 5 folds = 350 |
| Zone-Split LGB | 2 horizons × 2 zones × 20 seeds × 5 folds = 400 |
| **Total** | **1750** |

In [1]:
!pip install -q scikit-survival

import numpy as np
import pandas as pd
import warnings
import os
import time as timer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.util import Surv

warnings.filterwarnings('ignore')
np.random.seed(42)

HORIZONS_PRED = np.array([12, 24, 48, 72], dtype=float)

# Proven config
POWER_CAL_24 = 1.1
STRAT_THR = 5000  # near/far gate

# === Distance-stratified blend weights (from 0.9716 notebook v4 - best LB) ===
# Near zone (<5km): GBSA dominates timing
W_GBSA_NEAR_12 = 0.76;  W_COX_NEAR_12 = 0.12;  W_LGB_NEAR_12 = 0.10;  W_REM_NEAR_12 = 0.02
W_GBSA_NEAR_24 = 0.82;  W_COX_NEAR_24 = 0.14;  W_LGB_NEAR_24 = 0.02;  W_REM_NEAR_24 = 0.02
W_GBSA_NEAR_48 = 0.73;  W_COX_NEAR_48 = 0.16;  W_LGB_NEAR_48 = 0.08;  W_REM_NEAR_48 = 0.03
# Far zone (>=5km): CoxPH and LGB get more weight
W_GBSA_FAR_24 = 0.62;  W_COX_FAR_24 = 0.25;  W_LGB_FAR_24 = 0.07;  W_REM_FAR_24 = 0.06
W_GBSA_FAR_48 = 0.35;  W_COX_FAR_48 = 0.22;  W_LGB_FAR_48 = 0.37;  W_REM_FAR_48 = 0.06

# Seeds
GBSA_SEEDS = (
    123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31, 2024, 2077, 3077, 123456, 654321,
)
COX_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033)
LGB_SEEDS = (
    123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033,
    279, 239, 70, 77, 31, 2024, 2077, 3077, 123456, 654321,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 12.4 MB/s eta 0:00:00


In [2]:
def locate_datasets():
    train_path, test_path = 'train.csv', 'test.csv'
    for search_root in ['/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26', '../input']:
        if os.path.exists(search_root):
            for root, _, files in os.walk(search_root):
                if 'train.csv' in files: train_path = os.path.join(root, 'train.csv')
                if 'test.csv' in files: test_path = os.path.join(root, 'test.csv')
    return train_path, test_path

train_path, test_path = locate_datasets()
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print(f'Training: {len(train_df)} samples, Test: {len(test_df)} samples')

Training: 221 samples, Test: 95 samples


In [3]:
def create_features(df):
    result = df.copy()
    dist = result['dist_min_ci_0_5h'].clip(lower=1)
    speed = result['closing_speed_m_per_h']
    perimeters = result['num_perimeters_0_5h']
    area_first = result['area_first_ha']
    result['log_distance'] = np.log1p(dist)
    result['inv_distance'] = 1 / (dist / 1000 + 0.1)
    result['inv_distance_sq'] = result['inv_distance'] ** 2
    result['sqrt_distance'] = np.sqrt(dist)
    result['dist_km'] = dist / 1000
    result['dist_km_sq'] = (dist / 1000) ** 2
    result['dist_rank'] = dist.rank(pct=True)
    fire_radius = np.sqrt(area_first * 10000 / np.pi)
    result['fire_radius_km'] = fire_radius / 1000
    result['radius_to_dist'] = fire_radius / dist
    result['area_to_dist_ratio'] = area_first / (dist / 1000 + 0.1)
    result['log_area_dist_ratio'] = np.log1p(area_first) - np.log1p(dist)
    result['has_movement'] = (perimeters > 1).astype(float)
    closing_pos = speed.clip(lower=0)
    result['eta_hours'] = np.where(closing_pos > 0.01, dist / closing_pos, 9999).clip(max=9999)
    result['log_eta'] = np.log1p(result['eta_hours'].clip(0, 9999))
    radial_growth = result['radial_growth_rate_m_per_h'].clip(lower=0)
    effective_closing = closing_pos + radial_growth
    result['effective_closing_speed'] = effective_closing
    result['eta_effective'] = np.where(effective_closing > 0.01, dist / effective_closing, 9999).clip(max=9999)
    result['threat_score'] = result['alignment_abs'] * speed / np.log1p(dist)
    result['fire_urgency'] = perimeters * speed
    result['growth_intensity'] = result['area_growth_rate_ha_per_h'] * perimeters
    result['zone_near'] = (dist < STRAT_THR).astype(float)
    result['zone_far'] = (dist >= STRAT_THR).astype(float)
    result['is_summer'] = result['event_start_month'].isin([6, 7, 8]).astype(float)
    result['is_afternoon'] = ((result['event_start_hour'] >= 12) & (result['event_start_hour'] < 20)).astype(float)
    # Zone-specific ranks
    near_mask = dist < STRAT_THR
    result['near_speed_rank'] = 0.0
    if near_mask.sum() > 0:
        result.loc[near_mask, 'near_speed_rank'] = speed[near_mask].rank(pct=True)
    result['far_threat_rank'] = 0.0
    far_mask = ~near_mask
    if far_mask.sum() > 0:
        result.loc[far_mask, 'far_threat_rank'] = result.loc[far_mask, 'threat_score'].rank(pct=True)
    drop_cols = ['relative_growth_0_5h', 'projected_advance_m', 'centroid_displacement_m',
                 'centroid_speed_m_per_h', 'closing_speed_abs_m_per_h', 'area_growth_abs_0_5h']
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    return result

train_processed = create_features(train_df)
test_processed = create_features(test_df)

In [4]:
def get_surv_predictions(model, X):
    surv_fns = model.predict_survival_function(X)
    preds = np.empty((len(surv_fns), len(HORIZONS_PRED)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i, :] = fn(np.clip(HORIZONS_PRED, t_min, t_max))
    return 1.0 - preds

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(float)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t))
    for i, t in enumerate(unique_t):
        at_risk = (times >= t).sum()
        censored_at_t = ((times == t) & (events == 0)).sum()
        if at_risk > 0: surv[i] = 1 - censored_at_t / at_risk
        if i > 0: surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side='right') - 1
        return max(surv[idx], 0.01) if idx >= 0 else 1.0
    weights = np.ones(len(times))
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon: weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon: weights[i] = 1.0 / G(horizon)
    return weights

def enforce_monotonicity(preds):
    result = np.clip(preds, 0, 1)
    for i in range(1, result.shape[1]):
        result[:, i] = np.maximum(result[:, i], result[:, i-1])
    return result

## Model 1 — GBSA (10 configs × 20 seeds × 5-fold)

In [5]:
X_surv_train = train_df.drop(columns=['event_id', 'event', 'time_to_hit_hours'])
X_surv_test = test_df.drop(columns=['event_id'])
y_surv = Surv.from_arrays(event=train_df['event'].astype(bool), time=train_df['time_to_hit_hours'])
event_values = train_df['event'].values
time_values = train_df['time_to_hit_hours'].values
dist_test = test_df['dist_min_ci_0_5h'].values
dist_train = train_df['dist_min_ci_0_5h'].values
near_train = dist_train < STRAT_THR
near_test = dist_test < STRAT_THR

gbsa_configs = [
    {'learning_rate': 0.01,  'subsample': 0.7,  'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.01,  'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 15, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.01,  'subsample': 0.6,  'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.005, 'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 2000, 'dropout_rate': 0.0},
    {'learning_rate': 0.01,  'subsample': 0.85, 'max_depth': 3, 'min_samples_leaf': 20, 'min_samples_split': 3, 'n_estimators': 1400, 'dropout_rate': 0.0},
    {'learning_rate': 0.008, 'subsample': 0.75, 'max_depth': 2, 'min_samples_leaf': 15, 'min_samples_split': 4, 'n_estimators': 1500, 'dropout_rate': 0.0},
    {'learning_rate': 0.015, 'subsample': 0.70, 'max_depth': 3, 'min_samples_leaf': 10, 'min_samples_split': 3, 'n_estimators': 1000, 'dropout_rate': 0.0},
    {'learning_rate': 0.005, 'subsample': 0.90, 'max_depth': 3, 'min_samples_leaf': 18, 'min_samples_split': 5, 'n_estimators': 2500, 'dropout_rate': 0.0},
    {'learning_rate': 0.01,  'subsample': 0.80, 'max_depth': 4, 'min_samples_leaf': 12, 'min_samples_split': 3, 'n_estimators': 1200, 'dropout_rate': 0.0},
    {'learning_rate': 0.02,  'subsample': 0.65, 'max_depth': 3, 'min_samples_leaf': 10, 'min_samples_split': 3, 'n_estimators': 800,  'dropout_rate': 0.0},
]

test_gbsa = np.zeros((len(X_surv_test), 4))
t0 = timer.time()
print(f'GBSA: {len(gbsa_configs)} configs x {len(GBSA_SEEDS)} seeds x 5-fold')

for cfg_idx, cfg in enumerate(gbsa_configs):
    cfg_test = np.zeros((len(X_surv_test), 4))
    for seed in GBSA_SEEDS:
        seed_test = np.zeros((len(X_surv_test), 4))
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_surv_train, event_values):
            m = GradientBoostingSurvivalAnalysis(**{**cfg, 'random_state': seed})
            m.fit(X_surv_train.iloc[tr_idx], y_surv[tr_idx])
            seed_test += get_surv_predictions(m, X_surv_test) / 5
        cfg_test += seed_test / len(GBSA_SEEDS)
    test_gbsa += cfg_test / len(gbsa_configs)
    print(f'  cfg {cfg_idx+1}/{len(gbsa_configs)} done [{(timer.time()-t0)/60:.1f}m]')

# PowerCal 24h
test_gbsa[:, 1] = np.clip(test_gbsa[:, 1] ** POWER_CAL_24, 0, 1)
print(f'GBSA done. PowerCal 24h alpha={POWER_CAL_24}')

GBSA: 10 configs x 20 seeds x 5-fold
  cfg 1/10 done [2.2m]
  cfg 2/10 done [4.6m]
  cfg 3/10 done [6.8m]
  cfg 4/10 done [10.8m]
  cfg 5/10 done [13.6m]
  cfg 6/10 done [16.3m]
  cfg 7/10 done [18.2m]
  cfg 8/10 done [23.2m]
  cfg 9/10 done [25.7m]
  cfg 10/10 done [27.1m]
GBSA done. PowerCal 24h alpha=1.1


## Model 2 — CoxPH (7 alphas × 10 seeds × 5-fold)
Linear survival model: different inductive bias from tree-based GBSA.

In [6]:
COX_FEATURES = [
    'dist_km', 'log_distance', 'inv_distance',
    'closing_speed_m_per_h', 'radial_growth_rate_m_per_h',
    'alignment_abs', 'threat_score', 'log_eta', 'eta_effective',
    'area_to_dist_ratio', 'fire_radius_km',
    'num_perimeters_0_5h', 'has_movement',
    'near_speed_rank', 'far_threat_rank',
    'is_summer', 'is_afternoon',
    'zone_near', 'zone_far',
]

avail_cox = [f for f in COX_FEATURES if f in train_processed.columns]
X_cox_train = train_processed[avail_cox].copy()
X_cox_test = test_processed[[f for f in avail_cox if f in test_processed.columns]].copy()

scaler = StandardScaler()
X_cox_train_sc = pd.DataFrame(scaler.fit_transform(X_cox_train), columns=X_cox_train.columns, index=X_cox_train.index)
X_cox_test_sc = pd.DataFrame(scaler.transform(X_cox_test), columns=X_cox_test.columns, index=X_cox_test.index)

cox_alphas = [0.001, 0.01, 0.05, 0.10, 0.50, 1.00, 2.00]
test_cox = np.zeros((len(X_cox_test_sc), 4))
t0_cox = timer.time()
print(f'CoxPH: {len(cox_alphas)} alphas x {len(COX_SEEDS)} seeds x 5-fold')

for alpha in cox_alphas:
    alpha_test = np.zeros((len(X_cox_test_sc), 4))
    for seed in COX_SEEDS:
        seed_test = np.zeros((len(X_cox_test_sc), 4))
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for tr_idx, va_idx in cv.split(X_cox_train_sc, event_values):
            try:
                m = CoxPHSurvivalAnalysis(alpha=alpha)
                m.fit(X_cox_train_sc.iloc[tr_idx], y_surv[tr_idx])
                seed_test += get_surv_predictions(m, X_cox_test_sc) / 5
            except:
                seed_test += 0.5 / 5
        alpha_test += seed_test / len(COX_SEEDS)
    test_cox += alpha_test / len(cox_alphas)
    print(f'  alpha={alpha} done')

print(f'CoxPH done [{(timer.time()-t0_cox)/60:.1f}m]')

CoxPH: 7 alphas x 10 seeds x 5-fold
  alpha=0.001 done
  alpha=0.01 done
  alpha=0.05 done
  alpha=0.1 done
  alpha=0.5 done
  alpha=1.0 done
  alpha=2.0 done
CoxPH done [0.3m]


## Zone-Split LGB IPCW (separate near/far classifiers)

In [7]:
NEAR_LGB_FEATURES = [
    'closing_speed_m_per_h', 'radial_growth_rate_m_per_h',
    'alignment_abs', 'num_perimeters_0_5h', 'area_growth_rate_ha_per_h',
    'eta_effective', 'log_eta', 'dist_km', 'threat_score',
    'near_speed_rank', 'event_start_hour', 'is_afternoon', 'fire_urgency',
    'area_first_ha', 'fire_radius_km',
]
FAR_LGB_FEATURES = [
    'dist_km', 'log_distance', 'inv_distance',
    'closing_speed_m_per_h', 'alignment_abs',
    'threat_score', 'log_eta', 'eta_effective',
    'area_to_dist_ratio', 'num_perimeters_0_5h',
    'far_threat_rank', 'is_summer', 'zone_far',
]

avail_near = [f for f in NEAR_LGB_FEATURES if f in train_processed.columns]
avail_far = [f for f in FAR_LGB_FEATURES if f in train_processed.columns]
X_near_tr = train_processed[avail_near]
X_near_te = test_processed[[f for f in avail_near if f in test_processed.columns]]
X_far_tr = train_processed[avail_far]
X_far_te = test_processed[[f for f in avail_far if f in test_processed.columns]]

near_lgb_cfgs = {
    24: {'max_depth': 2, 'learning_rate': 0.04, 'n_estimators': 200,
         'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_samples': 4,
         'reg_alpha': 0.5, 'reg_lambda': 1.5, 'num_leaves': 4},
    48: {'max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 150,
         'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_samples': 3,
         'reg_alpha': 0.3, 'reg_lambda': 1.0, 'num_leaves': 4},
}
far_lgb_cfgs = {
    24: {'max_depth': 2, 'learning_rate': 0.03, 'n_estimators': 200,
         'subsample': 0.7, 'colsample_bytree': 0.7, 'min_child_samples': 8,
         'reg_alpha': 1.0, 'reg_lambda': 3.0, 'num_leaves': 4},
    48: {'max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 150,
         'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_samples': 6,
         'reg_alpha': 0.5, 'reg_lambda': 2.0, 'num_leaves': 4},
}

lgb_near_test, lgb_far_test = {}, {}
print(f'Zone-Split LGB: {len(LGB_SEEDS)} seeds per zone')

for horizon in [24, 48]:
    y_bin, mask = make_binary_target(time_values, event_values, horizon)
    valid_idx = np.where(mask)[0]
    for zone, cfg_d, X_tr, X_te, test_d in [
        ('near', near_lgb_cfgs, X_near_tr, X_near_te, lgb_near_test),
        ('far', far_lgb_cfgs, X_far_tr, X_far_te, lgb_far_test),
    ]:
        cfg = cfg_d[horizon]
        all_test = np.zeros(len(X_te))
        for seed in LGB_SEEDS:
            ipcw_w_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
            mf = lgb.LGBMClassifier(**cfg, objective='binary', random_state=seed, verbose=-1)
            mf.fit(X_tr.iloc[valid_idx], y_bin[valid_idx], sample_weight=ipcw_w_full)
            all_test += mf.predict_proba(X_te)[:, 1]
        test_d[horizon] = all_test / len(LGB_SEEDS)
    print(f'  {horizon}h done')

print('Zone-split LGB done.')

Zone-Split LGB: 20 seeds per zone
  24h done
  48h done
Zone-split LGB done.


## Distance-Stratified Blend + 72h=1.0 + Submit

In [8]:
def zone_blend(gbsa, cox, lgb_near, lgb_far, near_mask,
               wg_n, wc_n, wl_n, wr_n, wg_f, wc_f, wl_f, wr_f):
    """Blend with different weights for near and far zones."""
    near = wg_n * gbsa + wc_n * cox + wl_n * lgb_near + wr_n * 0.5  # remainder uses 0.5
    far = wg_f * gbsa + wc_f * cox + wl_f * lgb_far + wr_f * 0.5
    return np.where(near_mask, near, far)

test_blend = np.zeros_like(test_gbsa)

# 12h: near uses full blend, far uses GBSA only (most reliable)
test_blend[:, 0] = zone_blend(
    test_gbsa[:, 0], test_cox[:, 0], lgb_near_test[24], lgb_far_test[24], near_test,
    W_GBSA_NEAR_12, W_COX_NEAR_12, W_LGB_NEAR_12, W_REM_NEAR_12,
    1.0, 0.0, 0.0, 0.0)  # far: 100% GBSA for 12h

# 24h: distance-stratified blend
test_blend[:, 1] = zone_blend(
    test_gbsa[:, 1], test_cox[:, 1], lgb_near_test[24], lgb_far_test[24], near_test,
    W_GBSA_NEAR_24, W_COX_NEAR_24, W_LGB_NEAR_24, W_REM_NEAR_24,
    W_GBSA_FAR_24, W_COX_FAR_24, W_LGB_FAR_24, W_REM_FAR_24)

# 48h: distance-stratified blend
test_blend[:, 2] = zone_blend(
    test_gbsa[:, 2], test_cox[:, 2], lgb_near_test[48], lgb_far_test[48], near_test,
    W_GBSA_NEAR_48, W_COX_NEAR_48, W_LGB_NEAR_48, W_REM_NEAR_48,
    W_GBSA_FAR_48, W_COX_FAR_48, W_LGB_FAR_48, W_REM_FAR_48)

# 72h: CONSTANT 1.0 (proven best across all runs — B72 ≈ 0)
test_blend[:, 3] = 1.0

# Enforce monotonicity
test_final = enforce_monotonicity(test_blend)

# Build submission
submission = pd.DataFrame({
    'event_id': test_df['event_id'],
    'prob_12h': test_final[:, 0],
    'prob_24h': test_final[:, 1],
    'prob_48h': test_final[:, 2],
    'prob_72h': test_final[:, 3],
})

output_path = '/kaggle/working/submission.csv' if os.path.isdir('/kaggle/working') else 'submission.csv'
submission.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print(f'Architecture: GBSA + CoxPH + Zone-Split LGB')
print(f'72h = constant 1.0 | PowerCal={POWER_CAL_24} | Distance-stratified blend')
print(submission.describe().round(4))

Saved: /kaggle/working/submission.csv
Architecture: GBSA + CoxPH + Zone-Split LGB
72h = constant 1.0 | PowerCal=1.1 | Distance-stratified blend
           event_id  prob_12h  prob_24h  prob_48h  prob_72h
count  9.500000e+01   95.0000   95.0000   95.0000      95.0
mean   5.695393e+07    0.2112    0.2892    0.3061       1.0
std    2.329721e+07    0.3213    0.3800    0.3985       0.0
min    1.066260e+07    0.0115    0.0439    0.0477       1.0
25%    4.027536e+07    0.0136    0.0464    0.0493       1.0
50%    5.480111e+07    0.0147    0.0477    0.0511       1.0
75%    7.536942e+07    0.4932    0.7414    0.8426       1.0
max    9.964946e+07    0.9866    0.9896    0.9896       1.0
